In [1]:
import copy
import re
import tempfile
import ase.io.cube
import ipywidgets as ipw
import nglview as nv
import numpy as np
import traitlets as tl
from aiida import orm, plugins
from cubehandler import Cube
from PIL import ImageColor
from ase.io import read
from ase import io
import nglview as nv
import ipywidgets as widgets

In [2]:
import os
from ipywidgets import Layout

cube_folder = sorted([name for name in os.listdir("./cubes") if name.endswith(".cube")])

cube_file_1 = widgets.Dropdown(
    options=cube_folder,
    value=cube_folder[0] if cube_folder else None,
    description="Molecule 1"
 )

cube_file_2 = widgets.Dropdown(
    options=cube_folder,
    value=cube_folder[1] if len(cube_folder) > 1 else (cube_folder[0] if cube_folder else None),
    description="Molecule 2"
 )

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./cubes/{cube_file_1.value}"
CUBE_PATH_2 = f"./cubes/{cube_file_2.value}"
acrolein14 = ase.io.read(CUBE_PATH_1)
acrolein15 = ase.io.read(CUBE_PATH_2)

molecule1 = nv.show_ase(acrolein14)
molecule2 = nv.show_ase(acrolein15)
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")

molecule1.layout = widgets.Layout(width="100%", flex="1 1 50%", min_width="0")
molecule2.layout = widgets.Layout(width="100%", flex="1 1 50%", min_width="0")

positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )
colour_scheme = widgets.Dropdown(
    options=["default", "electrostatic", "element"],
    value="default",
    description="Colour scheme",
 )

def get_surface_colors(scheme):
    if scheme == "default":
        return "#3366ff", "#ff4d4d"
    if scheme == "electrostatic":
        return "#f713d5", "#3366ff"
    if scheme == "element":
        return "#7a4dff", "#eeff00"
    return "#3366ff", "#ff4d4d"

positive_color, negative_color = get_surface_colors(colour_scheme.value)
status = widgets.HTML()

def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    global positive_color, negative_color
    positive_color, negative_color = get_surface_colors(colour_scheme.value)
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(cube_file_1.value, ext="cube")
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(cube_file_2.value, ext="cube")
    redraw_NTO_2()

for control in [positive_level, negative_level, colour_scheme]:
    control.observe(redraw_surfaces, names="value")

cube_file_1.observe(update_NTO_1, names="value")
cube_file_2.observe(update_NTO_2, names="value")

redraw_surfaces()

controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        widgets.VBox([colour_scheme], layout = Layout(border="1px solid black", width="40%")),
        status,
    ],
 )
molecule1_box = widgets.VBox([cube_file_1, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([cube_file_2, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))